# Final Assignment: Analyzing Historical Stock/Revenue Data and Building a Dashboard

**Course project:** IBM Data Science / Python Project for Data Science  
**Companies:** Tesla (TSLA) and GameStop (GME)

## Project objective

Retrieve historical stock prices, collect quarterly revenue data, clean the data, and visualize stock-price and revenue trends in interactive Plotly dashboards.

## Exercises

1. Question 1 — Extract Tesla stock data.
2. Question 2 — Scrape and clean Tesla revenue data.
3. Question 3 — Extract GameStop stock data.
4. Question 4 — Scrape and clean GameStop revenue data.
5. Question 5 — Build the Tesla dashboard.
6. Question 6 — Build the GameStop dashboard.

> **Data note:** This notebook requests live data, so results can change over time and an internet connection is required. Macrotrends may block automated requests (HTTP 403). The revenue helper first attempts the requested `requests` + `BeautifulSoup` scraping method; if blocked, it falls back to the company's quarterly income statement from Yahoo Finance via `yfinance`. The fallback is normalized to millions of USD so it can be plotted on the same scale.

## Setup — Install required libraries
Run this cell once in the notebook environment if the libraries are not already installed.

In [ ]:
%pip install -q yfinance pandas requests beautifulsoup4 lxml plotly

## Imports and shared helper functions
The helper below attempts to parse the quarterly revenue table from Macrotrends using `requests`, `BeautifulSoup`, and `pandas.read_html`. It includes a documented fallback for cases where the website blocks automated scraping.

In [ ]:
import re
import warnings
from io import StringIO

import pandas as pd
import requests
import yfinance as yf
from bs4 import BeautifulSoup
import plotly.graph_objects as go
from plotly.subplots import make_subplots

pd.set_option("display.max_columns", None)

REQUEST_HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/130.0.0.0 Safari/537.36"
    )
}


def _clean_revenue_table(raw_table):
    """Standardize a two-column quarterly revenue table to Date and Revenue.

    Revenue is expressed in millions of USD, matching Macrotrends' table units.
    """
    table = raw_table.iloc[:, :2].copy()
    table.columns = ["Date", "Revenue"]

    date_text = table["Date"].astype(str).str.strip()
    table["Date"] = pd.to_datetime(date_text, errors="coerce")
    table["Revenue"] = (
        table["Revenue"]
        .astype(str)
        .str.replace(r"[$,]", "", regex=True)
        .str.strip()
        .replace({"": pd.NA, "nan": pd.NA, "None": pd.NA, "-": pd.NA})
    )
    table["Revenue"] = pd.to_numeric(table["Revenue"], errors="coerce")
    table = table.dropna(subset=["Date", "Revenue"])
    table = table.drop_duplicates(subset=["Date"]).sort_values("Date")
    return table.reset_index(drop=True)


def _scrape_macrotrends_quarterly_revenue(ticker_symbol, company_slug):
    """Scrape quarterly revenue from Macrotrends with requests and BeautifulSoup."""
    url = (
        f"https://www.macrotrends.net/stocks/charts/"
        f"{ticker_symbol}/{company_slug}/revenue"
    )
    response = requests.get(url, headers=REQUEST_HEADERS, timeout=30)
    response.raise_for_status()

    soup = BeautifulSoup(response.text, "html.parser")
    if not soup.find("table"):
        raise ValueError("No HTML tables were found in the Macrotrends response.")

    # Parse HTML table elements found in the page.
    html_tables = pd.read_html(StringIO(str(soup)))
    candidates = []

    for raw_table in html_tables:
        if raw_table.shape[1] < 2:
            continue

        # The quarterly table uses ISO dates such as 2021-03-31.
        first_column = raw_table.iloc[:, 0].astype(str).str.strip()
        iso_date_count = first_column.str.fullmatch(r"\d{4}-\d{2}-\d{2}").sum()
        if iso_date_count < 3:
            continue

        cleaned = _clean_revenue_table(raw_table)
        if len(cleaned) >= 3:
            candidates.append(cleaned)

    if not candidates:
        raise ValueError(
            "Could not identify the quarterly revenue table in the Macrotrends HTML."
        )

    # Choose the candidate with the most valid quarterly observations.
    return max(candidates, key=len)


def _yfinance_quarterly_revenue_fallback(ticker_symbol):
    """Fallback: get quarterly revenue from Yahoo Finance via yfinance.

    Yahoo Finance reports statement amounts in USD; convert them to millions USD.
    """
    ticker = yf.Ticker(ticker_symbol)
    statement = None
    last_error = None

    for attribute in ("quarterly_income_stmt", "quarterly_financials"):
        try:
            candidate = getattr(ticker, attribute)
            if candidate is not None and not candidate.empty:
                statement = candidate
                break
        except Exception as exc:
            last_error = exc

    if statement is None or statement.empty:
        raise RuntimeError(
            f"Could not load quarterly financial statements for {ticker_symbol}."
        ) from last_error

    normalized_index = {
        re.sub(r"[^a-z]", "", str(index).lower()): index
        for index in statement.index
    }
    revenue_row = None
    for key in ("totalrevenue", "revenue", "salesrevenue"):
        if key in normalized_index:
            revenue_row = normalized_index[key]
            break
    if revenue_row is None:
        for key, original_index in normalized_index.items():
            if "revenue" in key:
                revenue_row = original_index
                break

    if revenue_row is None:
        raise RuntimeError(
            f"No revenue row was found in {ticker_symbol}'s quarterly statement."
        )

    revenue_series = pd.to_numeric(statement.loc[revenue_row], errors="coerce").dropna()
    fallback = pd.DataFrame({
        "Date": pd.to_datetime(revenue_series.index, errors="coerce"),
        "Revenue": revenue_series.to_numpy(dtype=float) / 1_000_000,
    })
    fallback = fallback.dropna(subset=["Date", "Revenue"])
    fallback = fallback.sort_values("Date").reset_index(drop=True)
    if fallback.empty:
        raise RuntimeError(f"Revenue data for {ticker_symbol} is empty after cleaning.")
    return fallback


def get_quarterly_revenue(ticker_symbol, company_slug):
    """Scrape Macrotrends first, then use Yahoo Finance if scraping is blocked."""
    try:
        revenue = _scrape_macrotrends_quarterly_revenue(
            ticker_symbol=ticker_symbol,
            company_slug=company_slug,
        )
        print(f"Revenue source: Macrotrends ({ticker_symbol})")
        return revenue
    except Exception as exc:
        warnings.warn(
            f"Macrotrends scraping failed for {ticker_symbol} ({exc}). "
            "Trying Yahoo Finance quarterly financials; values will be in millions of USD.",
            RuntimeWarning,
        )
        revenue = _yfinance_quarterly_revenue_fallback(ticker_symbol)
        print(f"Revenue source: Yahoo Finance fallback ({ticker_symbol}); values are in millions USD.")
        return revenue


## Question 1 — Extract Tesla stock data using `yfinance`
Create `tesla_data`, reset the index, and display the first five rows.

In [ ]:
# Extract Tesla historical stock prices.
tesla = yf.Ticker("TSLA")
tesla_data = tesla.history(period="max", auto_adjust=False)

if tesla_data.empty:
    raise RuntimeError("No Tesla stock data was returned. Check your internet connection or Yahoo Finance availability.")

tesla_data.reset_index(inplace=True)
tesla_data.head()

## Question 2 — Scrape Tesla revenue data
Create `tesla_revenue` with the required `Date` and `Revenue` columns, clean currency symbols/commas and missing rows, then display the last five rows.

In [ ]:
# Scrape Tesla quarterly revenue from Macrotrends; use a documented fallback if blocked.
tesla_revenue = get_quarterly_revenue("TSLA", "tesla")

# Confirm the required columns and display the final five rows.
tesla_revenue = tesla_revenue[["Date", "Revenue"]]
tesla_revenue.tail()

## Question 3 — Extract GameStop stock data using `yfinance`
Create `gme_data`, reset the index, and display the first five rows.

In [ ]:
# Extract GameStop historical stock prices.
gamestop = yf.Ticker("GME")
gme_data = gamestop.history(period="max", auto_adjust=False)

if gme_data.empty:
    raise RuntimeError("No GameStop stock data was returned. Check your internet connection or Yahoo Finance availability.")

gme_data.reset_index(inplace=True)
gme_data.head()

## Question 4 — Scrape GameStop revenue data
Create `gme_revenue` with the required `Date` and `Revenue` columns, clean currency symbols/commas and missing rows, then display the last five rows.

In [ ]:
# Scrape GameStop quarterly revenue; use a documented fallback if Macrotrends blocks access.
gme_revenue = get_quarterly_revenue("GME", "gamestop")

gme_revenue = gme_revenue[["Date", "Revenue"]]
gme_revenue.tail()

## Question 5 — Build the Tesla stock/revenue dashboard
The `make_graph` function creates two synchronized Plotly charts: daily closing stock price and quarterly revenue. Revenue units are millions of USD.

In [ ]:
def make_graph(stock_data, revenue_data, company_name):
    """Plot historical closing stock price and quarterly revenue in two panels."""
    stock = stock_data.copy()
    revenue = revenue_data.copy()

    # Normalize dates to timezone-naive timestamps and clean numeric columns.
    stock["Date"] = pd.to_datetime(stock["Date"], errors="coerce", utc=True).dt.tz_localize(None)
    revenue["Date"] = pd.to_datetime(revenue["Date"], errors="coerce", utc=True).dt.tz_localize(None)
    stock["Close"] = pd.to_numeric(stock["Close"], errors="coerce")
    revenue["Revenue"] = pd.to_numeric(revenue["Revenue"], errors="coerce")

    stock = stock.dropna(subset=["Date", "Close"]).sort_values("Date")
    revenue = revenue.dropna(subset=["Date", "Revenue"]).sort_values("Date")

    if stock.empty:
        raise ValueError(f"No valid share-price data is available for {company_name}.")
    if revenue.empty:
        raise ValueError(f"No valid revenue data is available for {company_name}.")

    fig = make_subplots(
        rows=2,
        cols=1,
        shared_xaxes=True,
        vertical_spacing=0.12,
        subplot_titles=(
            f"{company_name} Historical Share Price",
            f"{company_name} Quarterly Revenue",
        ),
    )

    fig.add_trace(
        go.Scatter(
            x=stock["Date"],
            y=stock["Close"],
            mode="lines",
            name="Closing Price",
            line={"width": 1.5},
        ),
        row=1,
        col=1,
    )

    fig.add_trace(
        go.Scatter(
            x=revenue["Date"],
            y=revenue["Revenue"],
            mode="lines+markers",
            name="Revenue",
            line={"width": 2},
        ),
        row=2,
        col=1,
    )

    fig.update_yaxes(title_text="Closing Price (USD)", row=1, col=1)
    fig.update_yaxes(title_text="Revenue (Millions USD)", row=2, col=1)
    fig.update_xaxes(title_text="Date", row=2, col=1)
    fig.update_layout(
        title=f"{company_name}: Historical Stock Price and Revenue",
        height=760,
        template="plotly_white",
        hovermode="x unified",
        showlegend=False,
        margin={"l": 70, "r": 30, "t": 90, "b": 60},
    )

    fig.show()
    return fig

# Display the Tesla dashboard.
tesla_dashboard = make_graph(tesla_data, tesla_revenue, "Tesla")

## Question 6 — Build the GameStop stock/revenue dashboard

In [ ]:
# Display the GameStop dashboard.
gme_dashboard = make_graph(gme_data, gme_revenue, "GameStop")

## Final checks
Run this cell to verify the four required DataFrames exist and that both revenue DataFrames contain the required columns.

In [ ]:
# Show dataset dimensions and verify the requested schema.
print("Tesla stock shape:", tesla_data.shape)
print("Tesla revenue shape:", tesla_revenue.shape)
print("GameStop stock shape:", gme_data.shape)
print("GameStop revenue shape:", gme_revenue.shape)

assert {"Date", "Close"}.issubset(tesla_data.columns)
assert {"Date", "Close"}.issubset(gme_data.columns)
assert tesla_revenue.columns.tolist() == ["Date", "Revenue"]
assert gme_revenue.columns.tolist() == ["Date", "Revenue"]
assert not tesla_revenue.empty
assert not gme_revenue.empty

print("All required DataFrames are present and the revenue schemas are valid.")